# 🎓 Evaluasi Kinerja Sistem Pengenalan Wajah Berbasis LBP dan SVM

**Laporan Eksperimen Pengujian Biometrik Lanjut**

Notebook ini merupakan panduan saintifik yang lengkap untuk demonstrasi uji coba *Computer Vision Traditional* LBP-SVM. Mencakup Analisa Komponen Vektor hingga Kurva Evaluasi Metrik Biometrik Standar Skripsi/Tesis.


In [ ]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import joblib
import cv2
from PIL import Image

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.decomposition import PCA
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__)) 
except NameError:
    BASE_DIR = os.path.abspath('')

if not os.path.exists(os.path.join(BASE_DIR, 'python_scripts')):
    BASE_DIR = r"C:\xampp\htdocs\mpg-hris"

DATASET_DIR = os.path.join(BASE_DIR, 'storage', 'app', 'private', 'face_datasets')
MODEL_DIR   = os.path.join(BASE_DIR, 'storage', 'app', 'private', 'face_models')
DEBUG_DIR   = os.path.join(BASE_DIR, 'storage', 'app', 'private', 'debug_verify')

DF_APPROVED = 1.5
DF_PENDING  = 0.5

scripts_path = os.path.join(BASE_DIR, 'python_scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
    
from lbp_features import extract_lbp_features, preprocess_face, FACE_SIZE, _lbp_uniform

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
profile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_profileface.xml')
print("✅ Lingkungan eksekusi Akademik Advanced berhasil dimuat.")


## 1. Arsitektur Klasifikasi: Load Model Multi-Class SVM
* **StandardScaler**: Mengubah mean pada dataset fitur LBP (1792 dimensi) ke nilai 0 berdeviasi 1.
* **SVM Decision Function**: Kami menggunakan kernel probabilistik margin L2 dengan evaluasi absolut jarak Hyperplane.


In [ ]:
def load_system_model():
    svm_path = os.path.join(MODEL_DIR, 'face_model.pkl')
    scaler_path = os.path.join(MODEL_DIR, 'face_scaler.pkl')
    labels_path = os.path.join(MODEL_DIR, 'face_labels.json')
    svm_m = joblib.load(svm_path)
    scaler_m = joblib.load(scaler_path)
    with open(labels_path, 'r') as f:
        lbl = json.load(f)
    return svm_m, scaler_m, lbl

svm, scaler, labels = load_system_model()
kelas_terdaftar = [str(c) for c in svm.classes_ if str(c) != 'unknown']

print(f"✅ Model klasifikasi berhasil diload ke memory. Identitas Aktif: {kelas_terdaftar}")


## 2. API Pipeline Wajah dan Crop Geometrik

In [ ]:
def detect_and_crop_face(gray):
    h, w = gray.shape
    if max(h, w) > 640:
        scale = 640.0 / max(h, w)
        gray  = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
        h, w  = gray.shape

    min_size = int(min(h, w) * 0.1)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(min_size, min_size))
    if len(faces) == 0:
        faces = face_cascade.detectMultiScale(gray, 1.05, 3, minSize=(min_size, min_size))
    if len(faces) == 0:
        faces = profile_cascade.detectMultiScale(gray, 1.1, 3, minSize=(min_size, min_size))

    if len(faces) == 0: return None
    faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
    (x, y, fw, fh) = faces[0]
    padding = int(max(fw, fh) * 0.1)
    x1, y1 = max(0, x - padding), max(0, y - padding)
    x2, y2 = min(w, x + fw + padding), min(h, y + fh + padding)
    crop = gray[y1:y2, x1:x2]
    return cv2.resize(crop, FACE_SIZE, interpolation=cv2.INTER_AREA)

def process_image_file(img_path):
    try:
        pil_img = Image.open(img_path)
        exif    = pil_img.getexif()
        orient  = exif.get(274, 1)
        if orient == 3:   pil_img = pil_img.rotate(180, expand=True)
        elif orient == 6: pil_img = pil_img.rotate(270, expand=True)
        elif orient == 8: pil_img = pil_img.rotate(90,  expand=True)
        img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    except Exception:
        img = cv2.imread(img_path)
    if img is None: return None, None, "Fail"
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    face = detect_and_crop_face(gray)
    if face is None: return img, None, "No Face"
    proc  = preprocess_face(face)
    feats = extract_lbp_features(proc)
    return face, feats, "OK"

def classify_secure(feats, expected_user_id):
    classes    = [str(c) for c in svm.classes_]
    feats_s    = scaler.transform([feats])
    df_values  = svm.decision_function(feats_s)[0]
    if expected_user_id in classes:
        idx       = classes.index(expected_user_id)
        user_df   = float(df_values[idx] if hasattr(df_values, '__len__') else df_values)
    else: user_df = -999.0

    df_sorted = sorted(enumerate(df_values), key=lambda x: -x[1])
    predicted = str(classes[df_sorted[0][0]])
    predicted_is_user = (predicted == expected_user_id)
    if predicted_is_user and user_df >= DF_APPROVED: status = "APPROVED"
    elif predicted_is_user and user_df >= DF_PENDING: status = "PENDING"
    else: status = "REJECTED"
    return {'predicted': predicted, 'user_df': user_df, 'status': status, 'match': status in ("APPROVED", "PENDING")}


## 3. Visualisasi Anatomi Feature Extraction LBP Penuh

In [ ]:
def detailed_visual_demo():
    sample_file = None
    target_user = None
    for d in os.listdir(DATASET_DIR):
        dir_path = os.path.join(DATASET_DIR, d)
        if os.path.isdir(dir_path):
            files = [f for f in os.listdir(dir_path) if f.startswith('raw_frame_') and f.endswith('.jpg')]
            if files:
                sample_file = os.path.join(dir_path, files[0])
                target_user = d
                break
    if not sample_file: return

    raw_img = cv2.imread(sample_file)
    raw_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    x, y, w, h = (0,0,0,0)
    if len(faces) > 0:
        faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces[0]

    face_raw, feats, msg = process_image_file(sample_file)
    if face_raw is None: return
    processed = preprocess_face(face_raw)
    lbp_map = _lbp_uniform(processed, n_points=8, radius=1)
    res = classify_secure(feats, expected_user_id=target_user)
    
    fig = plt.figure(figsize=(19, 11))
    fig.suptitle(f"Bedah Algoritma Vektor Wajah | Sampel Asli: User {target_user}", fontsize=20, fontweight='bold', y=0.98)
    
    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(raw_rgb)
    if w > 0:
        pad = int(max(w, h) * 0.1)
        ax1.add_patch(patches.Rectangle((max(0, x-pad), max(0, y-pad)), w+(pad*2), h+(pad*2), linewidth=3, edgecolor='yellow', facecolor='none'))
    ax1.set_title("1. Deteksi Wajah Geometris", fontsize=13)
    ax1.axis('off')
    
    ax2 = plt.subplot(2, 3, 2)
    ax2.imshow(face_raw, cmap='gray')
    ax2.set_title("2. Potongan Skala Abu 128px", fontsize=13)
    ax2.axis('off')
    
    ax3 = plt.subplot(2, 3, 3)
    ax3.imshow(processed, cmap='gray')
    ax3.set_title("3. Preprocessing Histogram CLAHE", fontsize=13)
    ax3.axis('off')
    
    ax4 = plt.subplot(2, 3, 4)
    ax4.imshow(lbp_map, cmap='viridis')
    ax4.set_title("4. Local Binary Map (LBP Texture)", fontsize=13)
    ax4.axis('off')
    
    ax5 = plt.subplot(2, 3, 5)
    ax5.imshow(processed, cmap='gray')
    step_x = 128 // 8
    step_y = 128 // 8
    for i in range(1, 8):
        ax5.axhline(i*step_y, color='lime', linestyle='dashed', alpha=0.8)
        ax5.axvline(i*step_x, color='lime', linestyle='dashed', alpha=0.8)
    ax5.set_title("5. Peta 8x8 Lokalisasi Vektor", fontsize=13)
    ax5.axis('off')
    
    ax6 = plt.subplot(2, 3, 6)
    ax6.bar(range(180), feats[:180], color='dodgerblue')
    ax6.set_title("6. Output Vektor Array LBP 1D (SVM Feed)", fontsize=13)
    
    plt.tight_layout()
    plt.show()

detailed_visual_demo()


## 4. Evaluasi Akurasi Dasar (Confusion Matrix)

In [ ]:
y_true = []
y_pred = []

for uid in kelas_terdaftar:
    folder = os.path.join(DATASET_DIR, uid)
    files  = sorted([f for f in os.listdir(folder) if f.startswith('raw_frame_') and f.endswith('.jpg')])
    for fname in files:
        face, feats, info = process_image_file(os.path.join(folder, fname))
        if feats is not None:
            result = classify_secure(feats, expected_user_id=uid)
            y_true.append(uid)
            y_pred.append(result['predicted'])

labels_in_data = sorted(list(set(y_true + y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=labels_in_data)

fig, ax = plt.subplots(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_in_data)
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title("Confusion Matrix Otentikasi SVM", fontweight='bold', pad=20)
plt.show()


## 5. Visualisasi Multivariabel 1792 Dimensi Jarak SVM (PCA 2D Cluster)
Jika anda bertanya *'Bagaimana cara SVM membedakan manusia satu dengan lainnya?'*, grafik ini jawabannya.
Sistem ini menggunakan algoritma dimensi **Principal Component Analysis (PCA)** untuk mereduksi `1792 Dimensi` ke 2 dimensi X dan Y agar bisa terlihat oleh kacamata penguji. 
Apabila setiap klasifikasi identitas mengerucut dan mengumpul secara berkelompok (tidak berpencar atau meledak acak), maka model Machine Learning kita valid secara Geometrik.

In [ ]:
X_data = []           # Array Matrix Fitur
y_labels = []         # Array Identitas Sah
genuine_scores = []   # Track skor keyakinan untuk label sebenarnya
impostor_scores = []  # Track invasi hacker ketika coba masuk account kita

for uid in kelas_terdaftar:
    folder = os.path.join(DATASET_DIR, uid)
    files  = sorted([f for f in os.listdir(folder) if f.startswith('raw_frame_') and f.endswith('.jpg')])
    for fname in files:
        face, feats, msg = process_image_file(os.path.join(folder, fname))
        if feats is not None:
            X_data.append(feats)
            y_labels.append(uid)
            
            # --- Perhitungan Skor Jarak untuk Grafik Histogram Lonceng dan EER Nantinya ---
            feats_s = scaler.transform([feats])
            df_values = svm.decision_function(feats_s)[0]
            for class_idx, predicted_uid in enumerate(svm.classes_):
                predicted_uid = str(predicted_uid)
                if predicted_uid == 'unknown': continue
                
                score = df_values[class_idx]
                if predicted_uid == uid:
                    genuine_scores.append(score)
                else:
                    impostor_scores.append(score)

# --- MULAI PLOT PCA ---
print("Melakukan faktorisasi kompresi dimensi PCA...")
pca = PCA(n_components=2)
components = pca.fit_transform(X_data)

df_pca = pd.DataFrame(data = components, columns = ['Komponen LBP Skala Utama 1', 'Komponen LBP Skala Utama 2'])
df_pca['Identitas Ground Truth'] = y_labels

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='Komponen LBP Skala Utama 1', y='Komponen LBP Skala Utama 2',
    hue='Identitas Ground Truth', data=df_pca, 
    palette=sns.color_palette("Set1", len(kelas_terdaftar)),
    s=120, alpha=0.9, edgecolor='white', linewidth=0.5
)
plt.title('Pemetaan Relasi Fitur Wajah Manusia SVM (Clustering Visual)', fontsize=15, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Database Pegawai', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 6. Histogram Lonceng Biometrik (*Genuine Score vs Impostor Score*)
Grafik penting otentikasi biometrik yang berfungsi melihat "Jarak Jembatan" (*Gap Wall*) ketebalan perlindungan Margin antara Wajah Pengguna Asli (*Genuine*) dan Penyusup Acak (*Impostor*).
Apabila Grafik Merah (*Impostor*) menembus dan bersinggungan erat menutupi bukit warna Hijau (*Authentic*), sistem tersebut dinilai rentan dan berbahaya!

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(genuine_scores, color='seagreen', label='Genuine Access (Orang Dalam Sah)', kde=True, stat="density", bins=20, alpha=0.5, edgecolor="none")
sns.histplot(impostor_scores, color='crimson', label='Impostor Access (Orang Lain Tersesat)', kde=True, stat="density", bins=20, alpha=0.5, edgecolor="none")

plt.axvline(x=DF_APPROVED, color='gold', linestyle='--', linewidth=3, label=f'Margin Titik Kunci Server: DF({DF_APPROVED})')

plt.title('Sebaran Probabilitas Decision Margin SVM Wajah', fontsize=15, fontweight='bold')
plt.xlabel('SVM Decision Margin Score / Angka Kepastian')
plt.ylabel('Kepadatan Distribusi Data (%)')
plt.legend(loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 7. Receiver Operating Characteristic (ROC) & Titik Evaluasi Otentikasi EER Ekstrim
- **Kurva Biru ROC (Gambar Kiri)**: Menggambarkan luasan area keakuratan (*Area Under Curve / AUC*), dinilai superlatif bilamana mendekati angka 1.00.
- **Titik Silang EER (Gambar Kanan)**: Adalah titik krusial persinggungan antara Resiko Alarm Palsu **(FAR)** dengan Resiko Salah Tolak Orang Nyata **(FRR)**. Secara akademis _Golden Standard Error Rate_ harus bernilai 0% ~ 1%. 

In [ ]:
# Label array: Asli (1) dan Palsu / Impostor (0)
y_binary_roc = [1] * len(genuine_scores) + [0] * len(impostor_scores)
y_scores_roc = genuine_scores + impostor_scores

fpr_roc, tpr_roc, thresholds_roc = roc_curve(y_binary_roc, y_scores_roc)
roc_auc = auc(fpr_roc, tpr_roc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ---- 1. ROC CURVE ----
ax1.plot(fpr_roc, tpr_roc, color='dodgerblue', lw=3, label=f'Support Vector ROC (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='darkgrey', lw=2, linestyle='--', label='Garis Lemparan Koin Acak (Keburukan)')
ax1.set_xlim([-0.05, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate (Rasio Kebocoran Penipu)', fontsize=12)
ax1.set_ylabel('True Positive Rate (Rasio Menyetujui Akun Sendiri)', fontsize=12)
ax1.set_title('Kesempurnaan Deteksi / ROC Curve', fontweight='bold', fontsize=14)
ax1.legend(loc='lower right')
ax1.grid(True, linestyle=':', alpha=0.7)

# ---- 2. EER (EQUAL ERROR RATE) CURVE ----
# Formula Dasar Biometrik FRR adalah (100% - Tingkat Akseptasi Kebenaran TPR / False Rejection)
frr = 1 - tpr_roc
far = fpr_roc

# Deteksi titik perpapasan/silang untuk mendapatkan Absolute Equal Error Rate (EER) Tengah
eer_index = np.argmin(np.abs(far - frr))
eer_threshold_val = thresholds_roc[eer_index]
eer_percent = far[eer_index]

ax2.plot(thresholds_roc, far, color='crimson', lw=2.5, label='FAR (Orang Asing Diterima / Bahaya)')
ax2.plot(thresholds_roc, frr, color='navy', lw=2.5, label='FRR (Orang Asli Ditolak Mesin)')
ax2.axvline(x=eer_threshold_val, color='gold', linestyle='--', linewidth=2, label=f'EER={eer_percent*100:.2f}% (Toleransi Error Mutlak)\nMargin Kritis di: {eer_threshold_val:.2f}')

ax2.set_xlim([-1.5, 3.5]) # Memfokuskan grafik ke pergeseran persimpangan Decision Values
ax2.set_ylim([-0.05, 1.05])
ax2.set_xlabel('SVM Operational Decision Boundary Margin Limit', fontsize=12)
ax2.set_ylabel('Level Bahaya Error Tolerance', fontsize=12)
ax2.set_title('Kesetimbangan Resiko Operasional: FAR vs FRR', fontweight='bold', fontsize=14)
ax2.legend(loc='upper right')
ax2.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()

print("=" * 60)
print(f" >>> AREA UNDER CURVE KUALITAS SVM MURNI: {roc_auc:.4f} <<< ")
print(" Titik AUC 1.000 memformulasikan Algoritma telah mengenali Identitas tanpa misklasifikasi sedetik pun di atas kertas.")
print(f" ✅ Equal Error Rate (EER) Absolut ditekan ke angka ekstrim: {eer_percent*100:.2f} % !!!")
print("=" * 60)


## 8. Verifikasi Kasus Lintas Domain (Kondisi Lapangan Aplikasi Mobile)
Penyelesaian *Domain Gap* untuk mendiagnosa jepretan kamera Low Quality langsung dari Flutter Front-End Apps.

In [ ]:
if not os.path.exists(DEBUG_DIR): print("Folder debug_verify log tidak ditemukan.")
else:
    debug_files = sorted([f for f in os.listdir(DEBUG_DIR) if f.endswith('.jpg') and f.startswith('user')])
    
    if len(debug_files) == 0: print("Berdasarkan pengumpulan lapangan belum ada data log dari Presensi App.")
    else:
        print(f"Ditemukan {len(debug_files)} Gambar Log Absen Harian Teruji Secara Domain Silang:\n")
        print(f"{'Nama File Uji':<32} {'Target':<7} {'Status':<10} {'Margin SVM':>10} {'Akurasi Jauh'}")
        print("-" * 80)
        
        real_total = 0
        real_correct = 0
        for fname in debug_files:
            expected_uid = fname.split('_')[0].replace('user', '')
            fpath        = os.path.join(DEBUG_DIR, fname)
            face, feats, msg = process_image_file(fpath)
            if feats is None: continue
            r = classify_secure(feats, expected_user_id=expected_uid)
            real_total += 1
            if r['match']: real_correct += 1
                
            mar = "✔️ Lolos Security" if r['match'] else "❌ Gagal (Wajah Ditolak/Noise)"
            print(f"{fname:<32} User {expected_uid:<2} {r['status']:<10} {r['user_df']:>10.3f}   {mar}")
            
        print("-" * 80)
        print(f"Keandalan Validasi Perbedaan Domain & Resolusi Ekstrem Kamera Mobile: {real_correct}/{real_total} ({(real_correct/real_total)*100 if real_total else 0:.1f}%)")
